# qwen25-7b-fp8-quant-pipeline

| | |
|---|---|
| **Type** | KFP v2 FP8 post-training quantization pipeline |
| **Model** | [Qwen/Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) |
| **Method** | `nvidia-modelopt[hf]` FP8 PTQ (weights + KV cache), calibrated on `cnn_dailymail` |

FP8 post-training quantization of Qwen2.5-7B-Instruct via nvidia-modelopt on the Miramar platform (DGX Spark).

**Development workflow:**
1. Edit `config.yaml` — set model ID, quantization params, eval tasks/thresholds
2. Write step logic in the `@dsl.component` cells below
3. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
4. Trigger **Deploy to KFP** workflow (or run **Compile & Submit** below)

**Pipeline DAG:**
```
download_model → baseline_eval → quantize → post_quant_eval → quantization_gate
```

> Every GPU stage runs serially via `.after()` — DGX Spark is single-GPU, so `baseline_eval`,
> `quantize`, and `post_quant_eval` cannot share the GPU concurrently.

## Step Development

Write step logic inside each `@dsl.component` function body.

- **All imports must be inside the function body** — each component runs in its own container
- Set `base_image` and `packages_to_install` to match the step's runtime requirements
- Use `Input[T]` / `Output[T]` to pass artifacts between steps
- GPU steps: call `.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")` in the pipeline cell

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

### download_model

Downloads the base model to the HF cache PVC before any eval or quantization step.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["huggingface_hub>=0.21"],
)
def download_model(
    base_model_id: str,
):
    import os
    from huggingface_hub import snapshot_download
    print(f"Downloading {base_model_id} to HF cache...")
    snapshot_download(
        repo_id=base_model_id,
        cache_dir="/root/.cache/huggingface/hub",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Download complete: {base_model_id}")

### baseline_eval

Evaluates the unquantized base model on `winogrande`/`arc_easy`/`piqa` (5-shot, per `config.yaml`)
before quantization. Records metrics to MLflow. Purpose: cheap insurance that the model loads and
runs on this ARM64/Blackwell hardware before spending time quantizing.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["vllm", "lm_eval[api]", "mlflow", "nvtx", "requests"],
)
def baseline_eval(
    base_model_id: str,
    tasks: list,
    num_fewshot: int,
    batch_size: int,
    port: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
):
    import json, pathlib, subprocess, sys, time, mlflow, nvtx, requests
    from lm_eval import simple_evaluate

    base_url = f"http://localhost:{port}"
    proc = subprocess.Popen([
        "vllm", "serve", base_model_id,
        "--host", "0.0.0.0", "--port", str(port),
        "--trust-remote-code",
        "--gpu-memory-utilization", "0.8",
    ])
    try:
        deadline = time.time() + 600
        healthy = False
        while time.time() < deadline:
            if proc.poll() is not None:
                raise RuntimeError(f"vllm serve exited early with code {proc.returncode}")
            try:
                if requests.get(f"{base_url}/health", timeout=5).ok:
                    healthy = True
                    break
            except requests.exceptions.ConnectionError:
                pass
            time.sleep(5)
        if not healthy:
            raise TimeoutError("vllm serve did not become healthy within 600s")

        with nvtx.annotate("baseline_eval"):
            results = simple_evaluate(
                model="local-completions",
                model_args=(
                    f"model={base_model_id},"
                    f"base_url={base_url}/v1/completions,"
                    f"num_concurrent=8,max_retries=3,tokenized_requests=True"
                ),
                tasks=tasks,
                num_fewshot=num_fewshot,
                batch_size=batch_size,
            )
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()

    scores = {t: results["results"][t]["acc,none"] for t in tasks}
    avg_acc = sum(scores.values()) / len(scores)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-baseline"):
        mlflow.log_params({"tasks": ",".join(tasks), "num_fewshot": num_fewshot})
        for t, s in scores.items():
            mlflow.log_metric(f"baseline_{t}_acc", s)
        mlflow.log_metric("baseline_avg_acc", avg_acc)

    out = {"baseline_avg_acc": avg_acc, **{f"baseline_{t}_acc": s for t, s in scores.items()}}
    pathlib.Path(metrics.path).write_text(json.dumps(out))
    print(f"Baseline avg accuracy: {avg_acc:.4f}")

### quantize

Runs FP8 PTQ (weights + KV cache) via the vendored NVIDIA Model-Optimizer `hf_ptq.py` script,
calibrated on `cnn_dailymail`. The Build cell inlines the three vendored files
(`vendor/hf_ptq.py`, `vendor/example_utils.py`, `vendor/cast_mxfp4_to_nvfp4.py`) as repr()-escaped
string constants at the `# <<< HF_PTQ_VENDOR_INJECT >>>` marker — they're written to a temp dir at
runtime and invoked via `subprocess.run`, since `hf_ptq.py` imports the other two unconditionally
at module level.

Exports flat HF-format checkpoint files directly to the shared `hf-model-cache` PVC at a
deterministic path (`quantization/{pipeline_name}/{run_id}`) — this is the exact contract a future
`serving-trt-fp8` sibling project's `deploy.yaml` will mount read-only and pass to `vllm serve`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "nvidia-modelopt[hf]==0.45.0",
        "compressed-tensors",
        "fire",
        "transformers_stream_generator",
        "zstandard",
        "mlflow",
        "nvtx",
    ],
)
def quantize(
    base_model_id: str,
    qformat: str,
    kv_cache_qformat: str,
    calib_dataset: str,
    calib_size: int,
    calib_seq: int,
    export_path: str,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
):
    import pathlib, subprocess, sys, tempfile, mlflow, nvtx

    # <<< HF_PTQ_VENDOR_INJECT >>>

    tmp_dir = pathlib.Path(tempfile.mkdtemp(prefix="hf_ptq_vendor_"))
    (tmp_dir / "hf_ptq.py").write_text(_HF_PTQ_SRC)
    (tmp_dir / "example_utils.py").write_text(_EXAMPLE_UTILS_SRC)
    (tmp_dir / "cast_mxfp4_to_nvfp4.py").write_text(_CAST_MXFP4_TO_NVFP4_SRC)

    pathlib.Path(export_path).mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, str(tmp_dir / "hf_ptq.py"),
        "--pyt_ckpt_path", base_model_id,
        "--qformat", qformat,
        "--kv_cache_qformat", kv_cache_qformat,
        "--calib_size", str(calib_size),
        "--calib_seq", str(calib_seq),
        "--dataset", calib_dataset,
        "--export_path", export_path,
        "--trust_remote_code",
        "--device", "cuda",
    ]

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-quantize"):
        mlflow.log_params({
            "qformat": qformat,
            "kv_cache_qformat": kv_cache_qformat,
            "calib_dataset": calib_dataset,
            "calib_size": calib_size,
            "calib_seq": calib_seq,
            "export_path": export_path,
        })
        with nvtx.annotate("quantize"):
            try:
                subprocess.run(cmd, check=True, cwd=str(tmp_dir))
            except subprocess.CalledProcessError as e:
                print(f"hf_ptq.py failed with exit code {e.returncode}", file=sys.stderr)
                raise

    print(f"Quantization complete. Exported to: {export_path}")

### post_quant_eval

Serves the quantized checkpoint with `vllm serve` and evaluates it via `lm_eval`'s
`local-completions` API model against the same tasks as `baseline_eval`. Always terminates the
vLLM subprocess in a `finally` block — an orphaned process holding the GPU would silently break
the gate step.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["vllm", "lm_eval[api]", "mlflow", "nvtx", "requests"],
)
def post_quant_eval(
    quantized_model_path: str,
    tasks: list,
    num_fewshot: int,
    batch_size: int,
    port: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
):
    import json, pathlib, subprocess, sys, time, mlflow, nvtx, requests
    from lm_eval import simple_evaluate

    base_url = f"http://localhost:{port}"
    proc = subprocess.Popen([
        "vllm", "serve", quantized_model_path,
        "--host", "0.0.0.0", "--port", str(port),
        "--trust-remote-code",
        "--gpu-memory-utilization", "0.8",
    ])
    try:
        deadline = time.time() + 600
        healthy = False
        while time.time() < deadline:
            if proc.poll() is not None:
                raise RuntimeError(f"vllm serve exited early with code {proc.returncode}")
            try:
                if requests.get(f"{base_url}/health", timeout=5).ok:
                    healthy = True
                    break
            except requests.exceptions.ConnectionError:
                pass
            time.sleep(5)
        if not healthy:
            raise TimeoutError("vllm serve did not become healthy within 600s")

        with nvtx.annotate("post_quant_eval"):
            results = simple_evaluate(
                model="local-completions",
                model_args=(
                    f"model={quantized_model_path},"
                    f"base_url={base_url}/v1/completions,"
                    f"num_concurrent=8,max_retries=3,tokenized_requests=True"
                ),
                tasks=tasks,
                num_fewshot=num_fewshot,
                batch_size=batch_size,
            )
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()

    scores = {t: results["results"][t]["acc,none"] for t in tasks}
    avg_acc = sum(scores.values()) / len(scores)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-postquant"):
        mlflow.log_params({"tasks": ",".join(tasks), "num_fewshot": num_fewshot})
        for t, s in scores.items():
            mlflow.log_metric(f"postquant_{t}_acc", s)
        mlflow.log_metric("postquant_avg_acc", avg_acc)

    out = {"postquant_avg_acc": avg_acc, **{f"postquant_{t}_acc": s for t, s in scores.items()}}
    pathlib.Path(metrics.path).write_text(json.dumps(out))
    print(f"Post-quant avg accuracy: {avg_acc:.4f}")

### quantization_gate

Compares baseline vs post-quantization accuracy against `accuracy_delta_threshold` from
`config.yaml`. Fails the pipeline if the drop is too large. Writes `gate_result.json`
unconditionally (pass or fail) to the shared PVC for human visibility — a future
`serving-trt-fp8` sibling project only cares about the `quantization/{pipeline_name}/{run_id}/`
checkpoint directory, not this file.

In [ ]:
@dsl.component(base_image="python:3.11-slim")
def quantization_gate(
    baseline_metrics: Input[Metrics],
    postquant_metrics: Input[Metrics],
    accuracy_delta_threshold: float,
    pipeline_name: str,
    base_model_id: str,
    qformat: str,
    run_id: str,
):
    import datetime, json, pathlib

    baseline = json.loads(pathlib.Path(baseline_metrics.path).read_text())
    postquant = json.loads(pathlib.Path(postquant_metrics.path).read_text())

    baseline_score = baseline["baseline_avg_acc"]
    quantized_score = postquant["postquant_avg_acc"]
    accuracy_delta = quantized_score - baseline_score
    eval_passed = accuracy_delta >= accuracy_delta_threshold

    print(f"Baseline score : {baseline_score:.4f}")
    print(f"Quantized score: {quantized_score:.4f}")
    print(f"Accuracy delta : {accuracy_delta:+.4f}  (threshold >= {accuracy_delta_threshold:+.4f})")
    print(f"Gate           : {'PASS' if eval_passed else 'FAIL'}")

    result = {
        "project": pipeline_name,
        "run_id": run_id,
        "base_model_id": base_model_id,
        "qformat": qformat,
        "eval_passed": bool(eval_passed),
        "baseline_score": round(baseline_score, 6),
        "quantized_score": round(quantized_score, 6),
        "accuracy_delta": round(accuracy_delta, 6),
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }
    out_dir = pathlib.Path(f"/root/.cache/huggingface/runs/{pipeline_name}/{run_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "gate_result.json").write_text(json.dumps(result, indent=2))
    print(f"Gate result written: {out_dir}/gate_result.json")

    if not eval_passed:
        raise RuntimeError(
            f"Quantization gate failed — accuracy delta {accuracy_delta:+.4f} "
            f"below threshold {accuracy_delta_threshold:+.4f}"
        )

### Pipeline

Wire the steps together. Defaults are read from `config.yaml` at import time.
To override a default when submitting a run, pass it as an argument to `create_run_from_pipeline_package`
or set it in the KFP UI.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_pipeline_name = "qwen25-7b-fp8-quant-pipeline"  # must match @dsl.pipeline(name=...)

from kfp import kubernetes as k8s_ext


@dsl.pipeline(
    name="qwen25-7b-fp8-quant-pipeline",
    description=_cfg["description"],
)
def pipeline(
    base_model_id: str = _cfg["model"]["id"],
    qformat: str = _cfg["quantization"]["qformat"],
    kv_cache_qformat: str = _cfg["quantization"]["kv_cache_qformat"],
    calib_dataset: str = _cfg["quantization"]["calib_dataset"],
    calib_size: int = _cfg["quantization"]["calib_size"],
    calib_seq: int = _cfg["quantization"]["calib_seq"],
    eval_tasks: list = _cfg["eval"]["tasks"],
    num_fewshot: int = _cfg["eval"]["num_fewshot"],
    eval_batch_size: int = _cfg["eval"]["batch_size"],
    accuracy_delta_threshold: float = _cfg["eval"]["accuracy_delta_threshold"],
    vllm_port: int = _cfg["eval"]["vllm_port"],
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "default",
):
    _quantized_model_path = f"/root/.cache/huggingface/quantization/{_pipeline_name}/{run_id}"

    dl = download_model(base_model_id=base_model_id)

    base_eval = baseline_eval(
        base_model_id=base_model_id,
        tasks=eval_tasks,
        num_fewshot=num_fewshot,
        batch_size=eval_batch_size,
        port=vllm_port,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    base_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("96G")
    base_eval.set_caching_options(enable_caching=False)
    base_eval.after(dl)
    # To profile with Nsight Operator: kubernetes.add_pod_label(base_eval, "nvidia-nsight-profile", "enabled")

    quant = quantize(
        base_model_id=base_model_id,
        qformat=qformat,
        kv_cache_qformat=kv_cache_qformat,
        calib_dataset=calib_dataset,
        calib_size=calib_size,
        calib_seq=calib_seq,
        export_path=_quantized_model_path,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    quant.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("96G")
    quant.set_caching_options(enable_caching=False)
    quant.after(base_eval)

    post_eval = post_quant_eval(
        quantized_model_path=_quantized_model_path,
        tasks=eval_tasks,
        num_fewshot=num_fewshot,
        batch_size=eval_batch_size,
        port=vllm_port,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
    )
    post_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("96G")
    post_eval.set_caching_options(enable_caching=False)
    post_eval.after(quant)

    gate = quantization_gate(
        baseline_metrics=base_eval.outputs["metrics"],
        postquant_metrics=post_eval.outputs["metrics"],
        accuracy_delta_threshold=accuracy_delta_threshold,
        pipeline_name=_pipeline_name,
        base_model_id=base_model_id,
        qformat=qformat,
        run_id=run_id,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = [
        "OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY",
        "WANDB_API_KEY", "LANGCHAIN_API_KEY", "NGC_API_KEY", "NVIDIA_API_KEY",
    ]
    _key_map = {k: k for k in _SECRET_KEYS}
    _HF_CACHE_PVC = "hf-model-cache"
    for _task in [dl, base_eval, quant, post_eval, gate]:
        k8s_ext.mount_pvc(_task, pvc_name=_HF_CACHE_PVC, mount_path="/root/.cache/huggingface")
        k8s_ext.use_secret_as_env(_task, _SECRET, _key_map)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell to regenerate `pipeline.py`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (useful for quick iteration).

In [ ]:
import kfp
from kfp import compiler, dsl
print(f'kfp {kfp.__version__}')

In [ ]:
# Compile the pipeline defined in pipeline.py
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
# Submit a run
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
# Poll run status
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)